# 1. 질문과 읽는 방법
기존 Human 로그에서 **기록된 입력 간격**, **상태 조건부 다음 행동**, **hold 결정**에 필요한 증거가 있는가?
위에서 아래로 실행한다. 코드 셀의 함수는 synthetic 테스트와 실제 source 감사가 공유한다.
모델 fit·dataset 생성·엔진 replay는 하지 않는다. 구조 검증과 정확한 재생 가능성은 다르다.
명세: `ai-harness/ml-experiments/experiments/human-action-data-readiness.md`.
입력 시도는 성공한 이동과 다를 수 있다. hold는 같은 pieceIndex 안에서도 조각을 바꾼다.

In [ ]:
import hashlib
import json
import math
import os
import tempfile
from collections import Counter, defaultdict
from pathlib import Path

EXPERIMENT_ID = "human-action-data-readiness"
DEFAULT_SOURCE = "data/datasets/tetris-ml-snapshots-backup-0908/raw/game-events.jsonl"
TASKS = ("logged_inter_action_timing", "next_action_with_state", "hold_decision")
STATUSES = ("usable", "limited", "unverified", "unusable")
ACTIONS = frozenset(("move_left", "move_right", "rotate", "rotate_ccw",
                     "soft_drop", "hard_drop", "hold"))

def require(condition, reason):
    if not condition:
        raise ValueError(reason)

def finite(value):
    return type(value) in (int, float) and math.isfinite(value)

def integer(value):
    return type(value) is int and value >= 0

def reject_constant(value):
    raise ValueError("NON_FINITE_JSON")

def json_bytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True,
                       allow_nan=False, indent=2) + "\n").encode("utf-8")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# 2. 작은 worked example
아래는 실데이터가 아닌 synthetic 입력이다. move→rotate→drop의 시각 차이가 120ms·180ms이면 **로그에 기록된 간격**은 분석할 수 있다.
이것만으로 조각 등장 후 반응시간이나 실제 성공을 알 수는 없다.
빈 hold, 기존 hold 교체, 같은 종류 조각 교체 모두 hold 직전 문맥이 필요하다.
실패한 hold와 성공 미확인 hold는 구분하고 최종 hold 슬롯에서 과거를 역산하지 않는다.

# 3. 환경·경로·원본 보호
`HUMAN_ACTION_SOURCE`는 repo-relative 파일, `HUMAN_ACTION_RUN_MODE`는 smoke/full이다.
`HUMAN_ACTION_REPORT`는 존재하지 않는 새 JSON 경로다. source·code·기존 artifact를 덮지 않는다.
출력은 임시 디렉터리 또는 이 실험의 ignored evidence 디렉터리만 허용한다.
full 설정 자체는 실행 승인이 아니다. 기본 실행은 smoke다.

In [ ]:
def find_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "ai-harness").is_dir() and (candidate / "package.json").is_file():
            return candidate.resolve()
    raise ValueError("REPOSITORY_ROOT_NOT_FOUND")

def checked_paths(root, source_name, report_name, mode):
    root = Path(root).resolve()
    relative = Path(source_name)
    require(not relative.is_absolute() and ".." not in relative.parts, "SOURCE_MUST_BE_REPO_RELATIVE")
    source = (root / relative).resolve(strict=True)
    require(source.is_relative_to(root) and source.is_file(), "SOURCE_OUTSIDE_REPOSITORY")
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    output = Path(report_name).absolute()
    require(output.suffix == ".json", "REPORT_MUST_BE_JSON")
    require(not output.exists() and not output.is_symlink(), "REPORT_ALREADY_EXISTS_OR_SYMLINK")
    parent = output.parent.resolve(strict=True)
    require(parent.is_dir(), "REPORT_PARENT_REQUIRED")
    canonical = parent / output.name
    evidence = root / "data/ml-experiments" / EXPERIMENT_ID
    temporary_roots = (Path(tempfile.gettempdir()).resolve(), Path("/tmp").resolve())
    require(canonical.is_relative_to(evidence) if canonical.is_relative_to(root)
            else any(canonical.is_relative_to(base) for base in temporary_roots), "REPORT_LOCATION_NOT_ALLOWED")
    require(canonical != source, "SOURCE_OUTPUT_COLLISION")
    # Existing hardlink leaves are rejected above, before opening either output path.
    return source, canonical


# 4. Streaming 파싱과 source 분포
DB envelope와 app event를 한 분석 표현으로 읽되 raw 내용을 바꾸지 않는다.
불명 규칙은 unspecified이며 20행만 보고 역사 규칙을 확정하지 않는다.
모든 JSON 행을 한 문자열로 합치지 않는다. 1차 pass에는 hash·작은 metadata만 보존한다.

In [ ]:
def normalize_event(raw):
    require(isinstance(raw, dict), "EVENT_OBJECT_REQUIRED")
    is_db = "event_type" in raw
    if is_db:
        payload = raw.get("payload")
        require(isinstance(payload, dict), "PAYLOAD_OBJECT_REQUIRED")
        mapping = {"eventType": "event_type", "schemaVersion": "schema_version",
                   "gameId": "game_id", "playerId": "player_id", "pieceIndex": "piece_index"}
        for inner, outer in mapping.items():
            if inner in payload and payload[inner] != raw.get(outer):
                raise ValueError("ENVELOPE_PAYLOAD_CONFLICT")
        event = {inner: raw.get(outer) for inner, outer in mapping.items()}
        require(isinstance(raw.get("event_id"), str) and raw["event_id"], "EVENT_ID_REQUIRED")
        require(raw.get("identity_version") == "identity-v1", "IDENTITY_VERSION_REQUIRED")
        identity = ("db", raw["identity_version"], raw["event_id"])
        trust = raw.get("trust_level", "unspecified")
    else:
        payload = raw
        event = {key: raw.get(key) for key in
                 ("eventType", "schemaVersion", "gameId", "playerId", "pieceIndex")}
        trust = "unspecified"
        identity = ("app", event["eventType"], event["gameId"],
                    event["playerId"], event["pieceIndex"])
    require(event["eventType"] in ("game_start", "placement_decision", "game_over"), "UNSUPPORTED_EVENT")
    require(type(event["schemaVersion"]) is int and event["schemaVersion"] in (1, 2, 3), "UNSUPPORTED_SCHEMA")
    require(isinstance(event["gameId"], str) and event["gameId"], "GAME_ID_REQUIRED")
    if event["eventType"] == "placement_decision":
        require(isinstance(event["playerId"], str) and event["playerId"], "PLAYER_ID_REQUIRED")
        require(integer(event["pieceIndex"]), "PIECE_INDEX_REQUIRED")
    else:
        require(event["playerId"] is None and event["pieceIndex"] is None, "LIFECYCLE_IDENTITY_INVALID")
    # Identity only contains validated scalar values.
    return identity, event, payload, str(trust)

def finite_float(value):
    parsed = float(value)
    require(math.isfinite(parsed), "NON_FINITE_JSON")
    return parsed

def iter_rows(source):
    with Path(source).open("rb") as handle:
        for number, line in enumerate(handle, 1):
            if not line.strip():
                yield number, None, "blank"
                continue
            try:
                raw = json.loads(line, parse_constant=reject_constant, parse_float=finite_float)
            except (ValueError, UnicodeError):
                yield number, None, "parse_failure"
                continue
            yield number, raw, "parsed"

def bucket(value, allowed):
    return value if isinstance(value, str) and value in allowed else ("unspecified" if value is None or value == "unspecified" else "other")

def event_metadata(event, payload, trust, line):
    board = payload.get("boardBefore")
    geometry = ("{}x{}".format(len(board), len(board[0]))
                if isinstance(board, list) and board and isinstance(board[0], list)
                else "unspecified")
    observation = payload.get("observation")
    sequence = payload.get("inputSequence")
    return {**event, "line": line, "source": bucket(payload.get("source"), ("human", "heuristic_ai")),
            "mode": bucket(payload.get("mode"), ("pve", "multi", "single")), "trust": bucket(trust, ("client_reported", "server_authoritative")),
            "ruleset": bucket(payload.get("gameplayRulesVersion"), ("legacy20-v1", "buffer20-v1")),
            "geometry": geometry,
            "stage": bucket(observation.get("stage"), ("lock_commit", "post_line_clear", "post_garbage_spawn")) if isinstance(observation, dict) else "unspecified",
            "hasHold": isinstance(sequence, list) and any(
                isinstance(item, dict) and item.get("action") == "hold" for item in sequence),
            "finishType": bucket(payload.get("finishType"), ("normal", "disconnect", "leave"))}


# 5. Identity와 game 경계
동일 identity·같은 내용은 한 번 센다. 내용이 다른 identity는 그 identity의 모든 행을 conflict로 옮긴다.
나중에 발견한 충돌 때문에 첫 행까지 unique valid에서 제외한다. 서로 다른 ID가 같은 piece를 가리키는 것도 별도 lineage 문제다.
piece 공백의 원인은 unknown이다. 별도 증거 없이 rate limit이라고 추정하지 않는다.

In [ ]:
def scan_source(source):
    counts = Counter()
    invalid = Counter()
    identities = {}
    for line, raw, kind in iter_rows(source):
        counts["lines"] += 1
        if kind != "parsed":
            counts[kind] += 1
            continue
        counts["parsed"] += 1
        try:
            key, event, payload, trust = normalize_event(raw)
        except ValueError as error:
            invalid[str(error)] += 1
            counts["invalid"] += 1
            continue
        digest = hashlib.sha256(json_bytes(raw)).hexdigest()
        if key not in identities:
            identities[key] = {"digest": digest, "rows": 1, "conflict": False,
                               "meta": event_metadata(event, payload, trust, line)}
        else:
            item = identities[key]
            item["rows"] += 1
            item["conflict"] |= item["digest"] != digest
    valid = [item["meta"] for item in identities.values() if not item["conflict"]]
    counts["uniqueValid"] = len(valid)
    counts["exactDuplicate"] = sum(item["rows"] - 1 for item in identities.values() if not item["conflict"])
    counts["identityConflictRows"] = sum(item["rows"] for item in identities.values() if item["conflict"])
    counts["nonblank"] = counts["parsed"] + counts["parse_failure"]
    require(counts["parsed"] == sum(counts[k] for k in
            ("uniqueValid", "exactDuplicate", "identityConflictRows", "invalid")), "PARSED_COUNT_MISMATCH")
    require(counts["lines"] == counts["nonblank"] + counts["blank"], "LINE_COUNT_MISMATCH")
    games = defaultdict(list)
    for meta in valid:
        games[meta["gameId"]].append(meta)
    distributions = {name: Counter() for name in
                     ("schemaVersion", "source", "mode", "trust", "ruleset", "geometry", "stage")}
    sources = Counter()
    for meta in valid:
        if meta["eventType"] == "placement_decision":
            for name in distributions:
                distributions[name][str(meta[name])] += 1
            sources[meta["source"] if meta["source"] in ("human", "heuristic_ai") else "other"] += 1
    counts["uniquePlacement"] = sum(sources.values())
    human = [m for m in valid if m["eventType"] == "placement_decision" and m["source"] == "human"]
    candidates = sorted({m["gameId"] for m in human})
    held = sorted({m["gameId"] for m in human if m["hasHold"]})
    smoke = candidates[:3]
    extra = next((game for game in held if game not in smoke), None)
    if extra is not None:
        smoke.append(extra)
    problems = defaultdict(set)
    lifecycle = {}
    pseudonyms = {}
    for game, rows in sorted(games.items()):
        starts = [m for m in rows if m["eventType"] == "game_start"]
        ends = [m for m in rows if m["eventType"] == "game_over"]
        rules = {m["ruleset"] for m in rows if m["ruleset"] != "unspecified"}
        geometries = {m["geometry"] for m in rows if m["geometry"] != "unspecified"}
        mixed = len(rules) > 1 or len(geometries) > 1
        lifecycle[game] = {"startCount": len(starts), "endCount": len(ends),
                           "completeNormal": len(starts) == len(ends) == 1 and ends[0]["finishType"] == "normal",
                           "mixedRuleset": mixed}
        people = sorted({m["playerId"] for m in rows if m["playerId"] is not None})
        pseudonyms[game] = {person: index + 1 for index, person in enumerate(people)}
        by_player = defaultdict(list)
        for m in rows:
            if m["eventType"] == "placement_decision" and m["source"] == "human":
                by_player[m["playerId"]].append(m)
                if mixed:
                    problems[m["line"]].add("MIXED_RULESET")
        for player_rows in by_player.values():
            pieces = defaultdict(list)
            for m in player_rows:
                pieces[m["pieceIndex"]].append(m["line"])
            previous = None
            for piece, lines in sorted(pieces.items()):
                if len(lines) > 1:
                    for line in lines:
                        problems[line].add("DUPLICATE_PIECE")
                if previous is not None and piece > previous + 1:
                    for line in lines:
                        problems[line].add("PIECE_GAP")
                previous = piece
    return {"counts": dict(counts), "invalidReasons": dict(invalid),
            "distributions": {key: dict(value) for key, value in distributions.items()},
            "placementBySource": dict(sources), "humanCount": len(human),
            "humanGameIds": candidates, "smokeGameIds": sorted(smoke),
            "validByLine": {m["line"]: m for m in human}, "games": lifecycle,
            "problems": problems, "pseudonyms": pseudonyms}


# 6. 입력 순서·시간
createdAt의 차이는 기록 시각의 차이다. 물리 키 간격이나 첫 반응시간으로 승격하지 않는다.
빈 배열·필드 누락·손상된 입력·시각 누락은 다르게 센다.
0ms는 허용하되 time reversal은 해당 구간의 시간 학습에 사용할 수 없다.
아래 함수는 실제 원본과 synthetic 예제가 공유한다.

In [ ]:
def inspect_segment(payload, meta, lineage=()):
    reasons = set(lineage)
    reasons.update(("TIMESTAMP_SEMANTICS_UNVERIFIED", "INPUT_OUTCOME_UNKNOWN"))
    reasons.add("START_STATE_UNVERIFIED" if "boardBefore" in payload and "activePiece" in payload
                else "START_STATE_MISSING")
    inputs = payload.get("inputSequence")
    counters = Counter()
    actions = Counter()
    intervals = []
    structural = bool(set(lineage) & {"MIXED_RULESET", "DUPLICATE_PIECE"})
    missing_time = False
    unknown_order = False
    sequence_bad = False
    reverse = False
    if not isinstance(inputs, list):
        inputs = []
        reasons.add("INPUT_SEQUENCE_MISSING_OR_INVALID")
        structural = True
    elif not inputs:
        reasons.add("INPUT_SEQUENCE_EMPTY")
        structural = True
    for index, item in enumerate(inputs):
        counters["inputElements"] += 1
        valid = (isinstance(item, dict) and isinstance(item.get("action"), str) and item["action"] in ACTIONS
                 and integer(item.get("pieceIndex")) and integer(item.get("tick")))
        if not valid:
            counters["invalidActionRecords"] += 1
            structural = True
            reasons.add("INVALID_ACTION_RECORD")
            continue
        counters["validActionRecords"] += 1
        action = item["action"]
        actions[action] += 1
        if item["pieceIndex"] != meta["pieceIndex"]:
            structural = True
            reasons.add("INPUT_PIECE_MISMATCH")
        if not integer(item.get("sequence")):
            unknown_order = True
            reasons.add("SEQUENCE_MISSING_OR_INVALID")
        timestamp = item.get("createdAt")
        if not finite(timestamp) or timestamp < 0:
            missing_time = True
            reasons.add("TIMESTAMP_MISSING_OR_INVALID")
        else:
            counters["timestampPresent"] += 1
        if action == "hold":
            counters["holdAttempts"] += 1
            # These are reported flags, not engine-verified success.
            if item.get("applied") is True:
                counters["holdReportedAppliedTrue"] += 1
            elif item.get("applied") is False:
                counters["holdReportedAppliedFalse"] += 1
            else:
                counters["holdOutcomeUnknown"] += 1
        if index:
            previous = inputs[index - 1]
            if isinstance(previous, dict):
                if integer(previous.get("sequence")) and integer(item.get("sequence")):
                    sequence_bad |= item["sequence"] <= previous["sequence"]
                before = previous.get("createdAt")
                if finite(before) and before >= 0 and finite(timestamp) and timestamp >= 0:
                    delta = timestamp - before
                    intervals.append(delta)
                    reverse |= delta < 0
    if sequence_bad:
        structural = True
        reasons.add("SEQUENCE_NON_INCREASING")
    if reverse:
        reasons.add("TIME_REVERSAL")
    if counters["holdAttempts"]:
        reasons.add("PRE_HOLD_CONTEXT_UNVERIFIED")
    else:
        reasons.add("HOLD_AVAILABILITY_UNVERIFIED")
    if structural or reverse:
        timing = "unusable"
    elif missing_time:
        timing = "unverified"
    elif len(inputs) < 2 or unknown_order:
        timing = "limited"
    else:
        timing = "usable"
    readiness = {"logged_inter_action_timing": timing,
                 "next_action_with_state": "unusable" if structural else "unverified",
                 "hold_decision": "unusable" if structural else "unverified"}
    # No schema here proves per-action start/hold context or replay; do not infer it.
    counters["holdSegments"] = int(counters["holdAttempts"] > 0)
    counters["zeroTimePairs"] = sum(delta == 0 for delta in intervals)
    counters["timePairs"] = len(intervals)
    return {"readiness": readiness, "reasons": sorted(reasons),
            "counts": dict(counters), "actions": dict(actions),
            "intervals": intervals,
            "snapshotFieldsPresent": {name: name in payload for name in
                                      ("boardBefore", "activePiece", "holdPiece", "nextQueue")},
            "actionExample": [item["action"] for item in inputs[:20]
                              if isinstance(item, dict) and isinstance(item.get("action"), str) and item["action"] in ACTIONS]}

def interval_summary(values):
    ordered = sorted(values)
    if not ordered:
        return {"count": 0, "min": None, "median": None, "p95": None, "max": None}
    return {"count": len(ordered), "min": ordered[0],
            "median": ordered[(len(ordered) - 1) // 2],
            "p95": ordered[math.ceil(.95 * len(ordered)) - 1], "max": ordered[-1]}


# 7. Hold·시작 상태·구간 경계
최종 hold 슬롯과 nextQueue는 lock 직전 값일 수 있다. hold 시도가 있다는 사실과
성공·직전 슬롯·사용 가능 여부는 별개의 증거다. 현재 기록 형식에는 그 전부가 보장되지 않으므로
hold_decision과 next_action_with_state는 replay를 포함한 별도 증거 검증 전까지 unverified다.
미완결 게임도 표시하되 그 안의 모든 기록을 일괄 폐기하지 않는다. 구간 간 공백을 이어 학습하지 않는다.

# 8. 과제별 적합성·한계
usable은 **기록된 간격 분석**처럼 좁게 정의한 과제에 대한 판정이다.
limited는 일부 정보만, unverified는 증거 부족, unusable은 해당 구간이 깨진 경우다.
reason은 중복될 수 있지만 readiness는 각 과제에서 한 구간당 하나다.
완성된 inputSequence 전체를 현재 판단 feature로 제공하지 않는다.

In [ ]:
def audit_source(source, source_label, mode="smoke"):
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    before = sha256_file(source)
    index = scan_source(source)
    selected = index["humanGameIds"] if mode == "full" else index["smokeGameIds"]
    require(selected, "NO_HUMAN_COHORT")
    selected_set = set(selected)
    totals = Counter()
    action_counts = Counter()
    reasons = Counter()
    readiness = {task: Counter({status: 0 for status in STATUSES}) for task in TASKS}
    intervals = []
    traces = []
    for line, raw, kind in iter_rows(source):
        meta = index["validByLine"].get(line)
        if meta is None or meta["gameId"] not in selected_set:
            continue
        require(kind == "parsed", "SOURCE_CHANGED_DURING_AUDIT")
        _, _, payload, _ = normalize_event(raw)
        item = inspect_segment(payload, meta, index["problems"].get(line, ()))
        totals["humanSegments"] += 1
        totals.update(item["counts"])
        action_counts.update(item["actions"])
        reasons.update(item["reasons"])
        intervals.extend(item["intervals"])
        for task, status in item["readiness"].items():
            readiness[task][status] += 1
        traces.append({"gameId": meta["gameId"], "playerNumber": index["pseudonyms"][meta["gameId"]][meta["playerId"]],
                       "pieceIndex": meta["pieceIndex"], "sourceLine": line,
                       "readiness": item["readiness"], "reasons": item["reasons"],
                       "snapshotFieldsPresent": item["snapshotFieldsPresent"],
                       "holdAttempts": item["counts"].get("holdAttempts", 0),
                       "actionExample": item["actionExample"]})
    after = sha256_file(source)
    require(before == after, "SOURCE_CHECKSUM_CHANGED")
    require(totals["inputElements"] == totals["validActionRecords"] + totals["invalidActionRecords"],
            "ACTION_COUNT_MISMATCH")
    require(all(sum(counts.values()) == totals["humanSegments"] for counts in readiness.values()),
            "READINESS_COUNT_MISMATCH")
    expected = sum(m["gameId"] in selected_set for m in index["validByLine"].values())
    require(totals["humanSegments"] == expected, "COHORT_COUNT_MISMATCH")
    if mode == "full":
        require(totals["humanSegments"] == index["humanCount"], "FULL_COHORT_MISMATCH")
    traces.sort(key=lambda row: (row["gameId"], row["playerNumber"], row["pieceIndex"], row["sourceLine"]))
    # Keep trace rows compact; board and original actions/timestamps are never copied.
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
            "status": "data_readiness_smoke_valid" if mode == "smoke" else "data_readiness_audit_complete",
            "runMode": mode, "fullCohortCovered": mode == "full",
            "policyDecisionEligible": False, "canonicalResultEligible": False,
            "dataDecisionEligible": mode == "full", "modelFitCount": 0,
            "replayStatus": "unverified", "sourceUnchanged": True,
            "source": {"path": source_label, "sha256": before, "bytes": Path(source).stat().st_size},
            "sourceScan": {key: index[key] for key in
                           ("counts", "invalidReasons", "distributions", "placementBySource", "humanCount")},
            "cohort": {"selection": "all_human_games" if mode == "full" else "first_three_plus_first_additional_hold_game",
                       "gameIds": selected, "games": {game: index["games"][game] for game in selected}},
            "detail": {"counts": dict(totals), "actionCounts": dict(action_counts),
                       "reasonCounts": dict(reasons), "readiness": {key: dict(value) for key, value in readiness.items()},
                       "loggedIntervalMs": interval_summary(intervals), "segments": traces},
            "limits": ["no_engine_replay", "timestamp_generation_semantics_unverified",
                       "no_first_reaction_time", "pre_hold_context_unverified",
                       "input_success_not_proven", "piece_gap_cause_unknown",
                       "no_training_dataset_or_model", "no_cross_source_merge"]}

def publish_report(report, output):
    output = Path(output)
    require(not output.exists() and not output.is_symlink(), "REPORT_ALREADY_EXISTS_OR_SYMLINK")
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=output.parent, prefix=".human-action-", delete=False) as handle:
            temporary = Path(handle.name)
            handle.write(json_bytes(report))
            handle.flush()
            os.fsync(handle.fileno())
        # Exclusive publication: a concurrently created destination is never overwritten.
        os.link(temporary, output)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

def run_from_environment():
    root = find_root()
    mode = os.environ.get("HUMAN_ACTION_RUN_MODE", "smoke")
    require(mode != "full" or "HUMAN_ACTION_REPORT" in os.environ, "FULL_REPORT_PATH_REQUIRED")
    source_name = os.environ.get("HUMAN_ACTION_SOURCE", DEFAULT_SOURCE)
    report_name = os.environ.get("HUMAN_ACTION_REPORT", "/private/tmp/human-action-data-readiness-smoke.json")
    source, output = checked_paths(root, source_name, report_name, mode)
    report = audit_source(source, source.relative_to(root).as_posix(), mode)
    publish_report(report, output)
    return report


# 9. 집계 불변식·checksum·report
parsed 행은 unique valid + exact duplicate + identity conflict rows + invalid로 분해된다.
input 원소는 valid/invalid로, Human 구간은 과제별 네 상태로 분해된다.
source 전체 scan과 최대 4게임 smoke 상세 cohort를 분리한다. full에서도 modelFitCount=0이고
policyDecisionEligible=false다. 이 감사의 dataDecisionEligible은 데이터 판단 자료의 자격만 뜻한다.
출력은 exclusive atomic publication으로 기존 파일을 보존한다.

In [ ]:
example_meta = {"pieceIndex": 5}
example_inputs = [{"action": action, "tick": index, "pieceIndex": 5,
                   "sequence": index + 1, "createdAt": timestamp}
                  for index, (action, timestamp) in enumerate(
                      (("move_left", 1000), ("rotate", 1120), ("hard_drop", 1300)))]
worked = inspect_segment({"inputSequence": example_inputs}, example_meta)
assert worked["intervals"] == [120, 180]
print("Synthetic only:", worked["readiness"], worked["intervals"])


In [ ]:
REPORT = run_from_environment()
print(json.dumps({"status": REPORT["status"], "runMode": REPORT["runMode"],
                  "sourceCounts": REPORT["sourceScan"]["counts"],
                  "detailCounts": REPORT["detail"]["counts"],
                  "readiness": REPORT["detail"]["readiness"]},
                 ensure_ascii=False, indent=2))


# 10. 결과·한계와 다음 작업
smoke는 구현 확인이며 전체 cohort의 적합성 판정이 아니다. full은 사용자가 전체 Human 구간을 감사한다.
기록된 간격을 사용할 수 있어도 반응시간·상태 조건부 행동·hold 전략을 학습할 수 있다는 뜻은 아니다.
다음 작업은 과제별 dataset 범위와 추가 수집 항목을 결정하는 것이다. 모델 학습·runtime 통합은 시작하지 않는다.